In [11]:
import pandas as pd
import numpy as np
import sklearn

df = pd.read_csv('https://raw.githubusercontent.com/ageron/handson-ml2/refs/heads/master/datasets/housing/housing.csv')

from sklearn.model_selection import train_test_split
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

x_train = train_set.drop("median_house_value", axis=1)
y = train_set["median_house_value"].copy()

x_num = x_train.drop("ocean_proximity", axis=1)

In [12]:
from sklearn.base import BaseEstimator, TransformerMixin

rooms_ix, bedrooms_ix, population_ix, households_ix = 3, 4, 5, 6

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, add_bedrooms_per_room = True):
        self.add_bedrooms_per_room = add_bedrooms_per_room
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        rooms_per_household = X[:, rooms_ix] / X[:, households_ix]
        population_per_household = X[:, population_ix] / X[:, households_ix]
        if self.add_bedrooms_per_room:
            bedrooms_per_room = X[:, bedrooms_ix] / X[:, rooms_ix]
            return np.c_[X, rooms_per_household, population_per_household, bedrooms_per_room]
        else:
            return np.c_[X, rooms_per_household, population_per_household]

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [16]:
num_pipline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('attribs_adder', CombinedAttributesAdder()),
    ('std_scaler', StandardScaler())
])
num_pipline.fit_transform(x_num)

array([[ 1.27258656, -1.3728112 ,  0.34849025, ..., -0.17491646,
         0.05137609, -0.2117846 ],
       [ 0.70916212, -0.87669601,  1.61811813, ..., -0.40283542,
        -0.11736222,  0.34218528],
       [-0.44760309, -0.46014647, -1.95271028, ...,  0.08821601,
        -0.03227969, -0.66165785],
       ...,
       [ 0.59946887, -0.75500738,  0.58654547, ..., -0.60675918,
         0.02030568,  0.99951387],
       [-1.18553953,  0.90651045, -1.07984112, ...,  0.40217517,
         0.00707608, -0.79086209],
       [-1.41489815,  0.99543676,  1.85617335, ..., -0.85144571,
        -0.08535429,  1.69520292]])

In [17]:
from sklearn.compose import ColumnTransformer

num_attribs = list(x_num)
cat_attribs = ['ocean_proximity']

full_pipeline = ColumnTransformer([
    ('num', num_pipline, num_attribs),
    ('cat', OneHotEncoder(), cat_attribs)
])

In [18]:
x_prepared = full_pipeline.fit_transform(x_train)

In [20]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(x_prepared, y)


LinearRegression()

In [21]:
test_data = x_train.sample(10)
test_data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity
814,-122.03,37.61,37.0,1383.0,259.0,808.0,241.0,4.0125,NEAR BAY
7619,-118.24,33.83,22.0,7368.0,1367.0,4721.0,1342.0,4.8438,<1H OCEAN
2986,-119.00,35.33,35.0,991.0,221.0,620.0,207.0,1.9417,INLAND
12041,-117.49,33.90,7.0,10235.0,2238.0,5271.0,2094.0,3.6071,INLAND
6789,-118.16,34.09,52.0,1722.0,448.0,1122.0,425.0,3.1204,<1H OCEAN
13646,-117.31,34.08,43.0,1697.0,387.0,1181.0,352.0,1.9234,INLAND
14696,-117.09,32.79,36.0,1529.0,266.0,683.0,260.0,4.0982,NEAR OCEAN
8228,-118.19,33.77,35.0,1574.0,603.0,820.0,514.0,1.2321,NEAR OCEAN
7288,-118.22,33.98,42.0,626.0,143.0,625.0,156.0,3.1250,<1H OCEAN
3748,-118.38,34.19,42.0,1308.0,289.0,950.0,302.0,2.7379,<1H OCEAN


In [22]:
test_label = y.loc[test_data.index]
test_label

,median_house_value
814,161400.0
7619,213100.0
2986,53800.0
12041,159100.0
6789,224000.0
13646,74600.0
14696,171200.0
8228,137500.0
7288,166300.0
3748,181500.0


In [26]:
test_data_prepared = full_pipeline.transform(test_data)
predicted_labels = lr_model.predict(test_data_prepared)
predicted_labels

array([228899.4493658 , 231632.90595632,  96555.04172629, 189123.5512029 ,
       224497.66941213,  88541.83102026, 233067.26855199, 195166.55247454,
       193857.69472054, 181414.74468942])

In [27]:
pd.DataFrame({'Predicted': predicted_labels, 'Actual': test_label})

,Predicted,Actual
814,228899.449366,161400.0
7619,231632.905956,213100.0
2986,96555.041726,53800.0
12041,189123.551203,159100.0
6789,224497.669412,224000.0
13646,88541.831020,74600.0
14696,233067.268552,171200.0
8228,195166.552475,137500.0
7288,193857.694721,166300.0
3748,181414.744689,181500.0
